# Problem 3

I used `attr`, `shar`, and `intel` as predictors.

In [ ]:
import numpy as np
import pandas as pd
from cmdstanpy import CmdStanModel

predictors = ['attr', 'shar', 'intel']
seed = 42

df = pd.read_csv('speed_data_data.csv')[['dec', *predictors]].dropna()
df['dec'] = df['dec'].astype(int)

rng = np.random.default_rng(seed)
order = rng.permutation(len(df))
split = int(0.8 * len(df))

train = df.iloc[order[:split]]
test = df.iloc[order[split:]]

means = train[predictors].mean()
sds = train[predictors].std(ddof=0)

x_train = ((train[predictors] - means) / sds).to_numpy()
x_test = ((test[predictors] - means) / sds).to_numpy()
y_train = train['dec'].to_numpy(dtype=int)

stan_data = {
    'N': len(train),
    'P': len(predictors),
    'x': x_train,
    'y': y_train.tolist(),
    'N_test': len(test),
    'x_test': x_test,
}

model = CmdStanModel(stan_file='speed_dating.stan')
fit = model.sample(
    data=stan_data,
    seed=seed,
    chains=4,
    parallel_chains=4,
    iter_warmup=500,
    iter_sampling=500,
    show_progress=False,
)

summary = fit.summary()
beta_draws = fit.stan_variable('beta')

def ci(values):
    return [float(np.quantile(values, 0.025)), float(np.quantile(values, 0.975))]

print('total complete cases:', len(df))
print('train:', len(train))
print('test:', len(test))
print('max Rhat:', float(summary['R_hat'].max()))
print('min bulk ESS:', float(summary['ESS_bulk'].min()))
print('min tail ESS:', float(summary['ESS_tail'].min()))

for i, name in enumerate(predictors, start=1):
    row = summary.loc[f'beta[{i}]']
    print(name, 'mean =', float(row['Mean']), 'CI =', ci(beta_draws[:, i - 1]))


23:57:39 - cmdstanpy - INFO - CmdStan start processing
23:57:39 - cmdstanpy - INFO - Chain [1] start processing
23:57:39 - cmdstanpy - INFO - Chain [2] start processing
23:57:39 - cmdstanpy - INFO - Chain [3] start processing
23:57:39 - cmdstanpy - INFO - Chain [4] start processing
23:57:41 - cmdstanpy - INFO - Chain [1] done processing
23:57:41 - cmdstanpy - INFO - Chain [4] done processing
23:57:41 - cmdstanpy - INFO - Chain [3] done processing
23:57:41 - cmdstanpy - INFO - Chain [2] done processing


total complete cases: 7265
train: 5812
test: 1453
max Rhat: 1.00566
min bulk ESS: 996.797
min tail ESS: 1090.88
attr mean = 1.13689 CI = [1.0524005775, 1.222955045]
shar mean = 0.675147 CI = [0.60051693575, 0.7511544924999999]
intel mean = -0.102987 CI = [-0.173068745, -0.0298929633]



After dropping missing values, there were 7,265 complete cases. I used 5,812 for training and 1,453 for testing. The model converged. The maximum Rhat was 1.006, the minimum bulk ESS was 996.80, and the minimum tail ESS was 1090.88.

The posterior mean for `attr` was 1.137 with a 95% credible interval of [1.052, 1.223]. The posterior mean for `shar` was 0.675 with a 95% credible interval of [0.601, 0.751]. The posterior mean for `intel` was -0.103 with a 95% credible interval of [-0.173, -0.030]. The strongest predictor was attractiveness because it had the largest positive coefficient.